<a href="https://colab.research.google.com/github/EstebanBotero03/Senalesysistemas/blob/main/PUNTO_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PUNTO 1
ESTEBAN BOTERO OROZCO



### Celda 1: Instalación de Dependencias 📦

Esta celda de código ejecuta el comando `!pip install` para instalar las librerías de Python requeridas para la aplicación. La opción `-q` (quiet) se utiliza para una instalación limpia, sin mostrar los mensajes detallados.

* **`streamlit`**: Es el framework principal sobre el que se construye la aplicación web interactiva.
* **`numpy`**: Se utiliza para realizar operaciones numéricas, especialmente en la creación de los vectores de tiempo y en los cálculos de los parámetros del sistema.
* **`scipy`**: Una librería crucial para la computación científica. En este script, se usa su módulo `signal` para definir funciones de transferencia, calcular las respuestas al impulso y al escalón, y generar los diagramas de Bode.
* **`matplotlib`**: Es la encargada de generar todas las visualizaciones gráficas, como los diagramas de polos y ceros, las respuestas temporales y los diagramas de Bode.
* **`pandas`**: Se emplea para crear y mostrar de forma ordenada la tabla de parámetros temporales de la respuesta al escalón.
* **`pyngrok`**: Esta herramienta crea un túnel público y seguro hacia la aplicación de Streamlit que se ejecuta en el entorno de Colab, permitiendo su acceso desde cualquier navegador web.





In [2]:
!pip install streamlit numpy scipy matplotlib pandas pyngrok -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.6 MB/s eta 0:00:00


### Celda 2: Código de la Aplicación de Análisis de Sistemas (`streamlit_app.py`) 🔬

Esta celda utiliza el comando `%%writefile` para guardar todo el código Python en un archivo llamado `streamlit_app.py`. Este archivo contiene la lógica completa del dashboard interactivo para el análisis de sistemas de segundo orden.

#### **Estructura del Código:**

1.  **Configuración y Funciones de Ayuda**:
    * Se inicia configurando la página de Streamlit (`st.set_page_config`).
    * Se definen dos funciones auxiliares clave:
        * `calculate_temporal_params()`: Esta función toma la respuesta al escalón de un sistema y calcula métricas importantes como el tiempo de levantamiento (`tr`), el sobreimpulso máximo (`Mp`), el tiempo de pico (`tp`) y el tiempo de establecimiento (`ts`).
        * `analyze_system()`: Es la función principal de visualización. Recibe una función de transferencia y genera un conjunto completo de análisis gráficos: diagrama de polos y ceros, diagrama de Bode, respuesta al impulso, respuesta al escalón y respuesta a la rampa. También invoca a `calculate_temporal_params` para mostrar la tabla de métricas.

2.  **Interfaz de Usuario y Parámetros del Sistema**:
    * La barra lateral (`st.sidebar`) es el centro de control. El usuario puede seleccionar el tipo de respuesta deseada (Subamortiguada, Crítica, Sobreamortiguada o Inestable).
    * Basado en la selección, se presentan deslizadores (`st.slider`) para ajustar el **factor de amortiguamiento (ζ)** y la **frecuencia natural (ωn)**.
    * Con estos dos parámetros, el script calcula los coeficientes para las funciones de transferencia canónicas de segundo orden, tanto en lazo abierto como en lazo cerrado.

3.  **Cálculos Físicos y Visualización Principal**:
    * Como un añadido práctico, el script calcula y muestra los valores estimados de los componentes para un sistema mecánico (masa-resorte-amortiguador) y un sistema eléctrico (circuito RLC) que corresponderían a los parámetros ζ y ωn seleccionados.
    * La interfaz principal está organizada en dos pestañas (`st.tabs`): "Análisis en Lazo Abierto" y "Análisis en Lazo Cerrado".
    * En cada pestaña, se muestra la ecuación matemática de la función de transferencia (formateada con `st.latex`) y se llama a la función `analyze_system` para mostrar todas las gráficas y datos correspondientes a ese sistema.




In [3]:
%%writefile streamlit_app.py
import streamlit as st
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import pandas as pd

# --- Page Configuration ---
st.set_page_config(
    page_title="Análisis de Sistemas de 2º Orden",
    page_icon="🔬",
    layout="wide",
)

# --- Matplotlib Style ---
plt.style.use('seaborn-v0_8-darkgrid')

# --- Helper Functions ---

def calculate_temporal_params(t, y_step, final_value):
    """Calcula los parámetros temporales de la respuesta al escalón."""
    params = {
        'Tiempo de Levantamiento (tr)': 'N/A',
        'Sobre-impulso Máximo (Mp)': 'N/A',
        'Tiempo de Pico (tp)': 'N/A',
        'Tiempo de Establecimiento (ts)': 'N/A'
    }

    if final_value < 1e-6: # Evita división por cero
        return params

    try:
        t_10 = t[np.where(y_step >= 0.1 * final_value)[0][0]]
        t_90 = t[np.where(y_step >= 0.9 * final_value)[0][0]]
        params['Tiempo de Levantamiento (tr)'] = f"{t_90 - t_10:.3f} s"
    except IndexError:
        pass

    max_val = np.max(y_step)
    if max_val > final_value:
        mp = ((max_val - final_value) / final_value) * 100
        params['Sobre-impulso Máximo (Mp)'] = f"{mp:.2f} %"
        params['Tiempo de Pico (tp)'] = f"{t[np.argmax(y_step)]:.3f} s"
    else:
        params['Sobre-impulso Máximo (Mp)'] = "0.00 %"

    try:
        settling_mask = np.abs(y_step - final_value) > 0.02 * final_value
        if np.any(settling_mask):
            last_out_of_bounds = np.where(settling_mask)[0][-1]
            params['Tiempo de Establecimiento (ts)'] = f"{t[last_out_of_bounds]:.3f} s"
        else:
             params['Tiempo de Establecimiento (ts)'] = f"{t[0]:.3f} s"
    except IndexError:
        pass

    return params


def analyze_system(tf, zeta, wn):
    """
    Genera y muestra todos los análisis para una función de transferencia dada.
    """
    col1, col2 = st.columns([1, 1.5])

    with col1:
        # 1. Diagrama de Polos y Ceros
        fig, ax = plt.subplots(figsize=(6, 5))
        poles = tf.poles
        zeros = tf.zeros
        if zeros.size > 0:
            ax.plot(np.real(zeros), np.imag(zeros), 'o', markersize=10, label='Ceros', color='blue')
        ax.plot(np.real(poles), np.imag(poles), 'x', markersize=10, mew=2, label='Polos', color='red')
        ax.grid(True)
        ax.set_title("Diagrama de Polos y Ceros", fontsize=14)
        ax.set_xlabel("Eje Real")
        ax.set_ylabel("Eje Imaginario")
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)
        ax.legend()
        st.pyplot(fig)

        # 2. Diagrama de Bode
        w, mag, phase = signal.bode(tf)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
        ax1.semilogx(w, mag, color='dodgerblue')
        ax1.set_ylabel("Magnitud (dB)")
        ax1.set_title("Diagrama de Bode", fontsize=14)
        ax2.semilogx(w, phase, color='darkorange')
        ax2.set_xlabel("Frecuencia (rad/s)")
        ax2.set_ylabel("Fase (grados)")
        st.pyplot(fig)

    with col2:
        if zeta > 0 and wn > 0:
            t_final = 8 / (zeta * wn) if zeta > 0.1 else 20 / wn
        else:
            t_final = 10

        t = np.linspace(0, t_final, 1000)

        # 3. Respuesta al Impulso
        t_imp, y_imp = signal.impulse(tf, T=t)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(t_imp, y_imp, label="Respuesta al Impulso", color='green')
        ax.set_title("Respuesta al Impulso", fontsize=14)
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Amplitud")
        ax.legend()
        st.pyplot(fig)

        # 4. Respuesta al Escalón
        t_step, y_step = signal.step(tf, T=t)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(t_step, y_step, label="Respuesta al Escalón", color='purple')
        ax.axhline(y_step[-1], color='gray', linestyle='--', label=f'Valor Final: {y_step[-1]:.2f}')
        ax.set_title("Respuesta al Escalón", fontsize=14)
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Amplitud")
        ax.legend()
        st.pyplot(fig)

        # 5. Parámetros temporales (solo para sistemas estables)
        if zeta >= 0 and np.all(tf.poles.real < 0):
            st.write("**Parámetros de la Respuesta al Escalón:**")
            params = calculate_temporal_params(t_step, y_step, y_step[-1])
            df_params = pd.DataFrame(params.items(), columns=['Parámetro', 'Valor'])
            st.table(df_params)
        else:
            st.warning("Los parámetros temporales no se calculan para sistemas inestables.")

        # 6. Respuesta a la Rampa
        num_ramp = tf.num
        den_ramp = np.polymul(tf.den, [1, 0])
        tf_ramp = signal.TransferFunction(num_ramp, den_ramp)
        t_ramp, y_ramp = signal.step(tf_ramp, T=t)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(t_ramp, y_ramp, label="Respuesta a la Rampa", color='brown')
        ax.plot(t_ramp, t_ramp, '--', label="Entrada Rampa Ideal", color='gray')
        ax.set_title("Respuesta a la Rampa", fontsize=14)
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Amplitud")
        ax.legend()
        st.pyplot(fig)


st.title("🔬 Dashboard de Simulación de Sistemas de 2º Orden")
st.markdown("Dashboard interactivo para el Parcial 2 de Señales y Sistemas.")

st.sidebar.title("Parámetros del Sistema")

response_type = st.sidebar.selectbox(
    "1. Seleccione el tipo de respuesta:",
    ('Subamortiguada', 'Amortiguamiento Crítico', 'Sobreamortiguada', 'Inestable'),
)

if response_type == 'Subamortiguada':
    zeta = st.sidebar.slider("2. Factor de Amortiguamiento (ζ)", 0.01, 0.99, 0.3, 0.01)
elif response_type == 'Amortiguamiento Crítico':
    zeta = 1.0
    st.sidebar.markdown("ζ = 1 (Fijo)")
elif response_type == 'Sobreamortiguada':
    zeta = st.sidebar.slider("2. Factor de Amortiguamiento (ζ)", 1.01, 5.0, 1.5, 0.01)
else: # Inestable
    zeta = st.sidebar.slider("2. Factor de Amortiguamiento (ζ)", -1.0, -0.01, -0.5, 0.01)

wn = st.sidebar.slider("3. Frecuencia Natural (ωn) [rad/s]", 1.0, 20.0, 5.0, 0.1)

num_ol = [wn**2]
den_ol = [1, 2 * zeta * wn, wn**2]
tf_ol = signal.TransferFunction(num_ol, den_ol)

num_cl = num_ol
den_cl = [1, 2 * zeta * wn, wn**2 + wn**2]
tf_cl = signal.TransferFunction(num_cl, den_cl)

st.sidebar.title("Valores Estimados de Componentes")

m = 1.0
k = m * wn**2
c = 2 * zeta * wn * m
st.sidebar.markdown("**Sistema Mecánico (m=1kg):**")
st.sidebar.markdown(f"- **Rigidez (k):** `{k:.2f} N/m`")
st.sidebar.markdown(f"- **Amortiguador (c):** `{c:.2f} Ns/m`")

C_val = 1e-6
L_val = 1 / (wn**2 * C_val)
if zeta > 0:
    R_val = (1 / (2 * zeta)) * np.sqrt(L_val / C_val)
else:
    R_val = np.inf

st.sidebar.markdown("**Sistema Eléctrico (C=1µF):**")
st.sidebar.markdown(f"- **Inductancia (L):** `{L_val * 1e3:.2f} mH`")
if R_val != np.inf:
    st.sidebar.markdown(f"- **Resistencia (R):** `{R_val:.2f} Ω`")
else:
     st.sidebar.markdown(f"- **Resistencia (R):** `Inestable (ζ < 0)`")

tab1, tab2 = st.tabs(["Análisis en Lazo Abierto", "Análisis en Lazo Cerrado"])

with tab1:
    st.header("Análisis del Sistema en Lazo Abierto: $G(s)$")
    st.latex(f"G(s) = \\frac{{{wn**2:.2f}}}{{s^2 + {2*zeta*wn:.2f}s + {wn**2:.2f}}}")
    analyze_system(tf_ol, zeta, wn)

with tab2:
    st.header("Análisis del Sistema en Lazo Cerrado: $T(s)$")
    zeta_cl = (tf_cl.den[1] / (2 * np.sqrt(tf_cl.den[2]))) if tf_cl.den[2] > 0 else 0
    st.latex(f"T(s) = \\frac{{{tf_cl.num[0]:.2f}}}{{s^2 + {tf_cl.den[1]:.2f}s + {tf_cl.den[2]:.2f}}}")
    analyze_system(tf_cl, zeta_cl, np.sqrt(tf_cl.den[2]))

Writing streamlit_app.py


### Celda 3: Ejecución y Despliegue de la Aplicación 🚀

Esta celda final es la responsable de poner en marcha la aplicación de Streamlit y hacerla accesible a través de internet.

1.  **Autenticación de Ngrok**:
    * El usuario debe proporcionar su token de autenticación personal de [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) en la variable `NGROK_AUTHTOKEN`. Este paso es fundamental para que `ngrok` pueda crear el túnel.

2.  **Ejecución de Streamlit**:
    * El comando `!nohup streamlit run streamlit_app.py --server.port 8501 &` inicia la aplicación. Se ejecuta en el puerto `8501` y en segundo plano (`&`), lo que permite que el notebook de Colab continúe disponible mientras la app está activa.

3.  **Creación del Túnel Público**:
    * La función `ngrok.connect(8501)` establece la conexión entre una URL pública generada por `ngrok` y el puerto local `8501` donde se está ejecutando la aplicación.

4.  **Acceso al Dashboard**:
    * El script concluye imprimiendo la **URL pública**. El usuario debe hacer clic en este enlace para abrir el dashboard interactivo en su navegador y comenzar el análisis del sistema.

In [5]:
from pyngrok import ngrok

# Pega aquí tu token de autenticación de ngrok
NGROK_AUTHTOKEN = "2zLBw00gDbYx9vnFZneHVhwzE62_UptNrhkYSDvxyYsuhwng"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Lanza la app de Streamlit en segundo plano en el puerto 8501
!nohup streamlit run streamlit_app.py --server.port 8501 &

# Crea el túnel público con pyngrok
public_url = ngrok.connect(8501)
print("¡Tu dashboard está en vivo!")
print("URL Pública:", public_url)

nohup: appending output to 'nohup.out'
¡Tu dashboard está en vivo!
URL Pública: NgrokTunnel: "https://e457d5f88861.ngrok-free.app" -> "http://localhost:8501"
